In [0]:
%sql
USE uk_train_ride.train_rise;

In [0]:
%sql
SELECT 
    Reason_for_Delay AS Reason_for_Cancellation,
    COUNT(DISTINCT Transaction_ID) AS Passengers,
    SUM(CASE WHEN Refund_Request = 'Yes' THEN 1 ELSE 0 END) AS Refund_Requests,
    ROUND(
        (SUM(CASE WHEN Refund_Request = 'Yes' THEN 1 ELSE 0 END) * 100.0) / COUNT(*),
        2
    ) AS Refund_Percentage,
    SUM(CASE WHEN Refund_Request = 'Yes' THEN Price ELSE 0 END) AS Total_Refunds
FROM railway
WHERE Journey_Status = 'Cancelled'
GROUP BY Reason_for_Delay
ORDER BY Passengers DESC;

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Define consistent color scheme
COLORS = {
    'primary': '#1f77b4',
    'secondary': '#ff7f0e', 
    'tertiary': '#2ca02c',
    'quaternary': '#d62728',
    'purple': '#9467bd',
    'brown': '#8c564b'
}

# Get cancellation data
df_cancel = spark.sql("""
SELECT 
    Reason_for_Delay AS Reason,
    COUNT(DISTINCT Transaction_ID) AS Passengers,
    SUM(CASE WHEN Refund_Request = 'Yes' THEN 1 ELSE 0 END) AS Refund_Requests,
    ROUND((SUM(CASE WHEN Refund_Request = 'Yes' THEN 1 ELSE 0 END) * 100.0) / COUNT(*), 1) AS Refund_Pct,
    SUM(CASE WHEN Refund_Request = 'Yes' THEN Price ELSE 0 END) AS Total_Refunds
FROM uk_train_ride.train_rise.railway
WHERE Journey_Status = 'Cancelled'
GROUP BY Reason_for_Delay
ORDER BY Passengers DESC
""").toPandas()

# Create 4-panel dashboard
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
fig.suptitle('Cancellation Analysis by Reason', fontsize=18, fontweight='bold')

# Plot 1: Cancelled Services by Reason
ax1 = fig.add_subplot(gs[0, 0])
colors1 = [COLORS['quaternary'], COLORS['secondary'], COLORS['purple'], 
          COLORS['brown'], COLORS['primary']]
bars1 = ax1.bar(range(len(df_cancel)), df_cancel['Passengers'],
               color=colors1, alpha=0.8)
ax1.set_xticks(range(len(df_cancel)))
ax1.set_xticklabels(df_cancel['Reason'], rotation=45, ha='right', fontsize=9)
ax1.set_title('Cancelled Services by Reason', fontsize=13, fontweight='bold')
ax1.set_ylabel('Number of Passengers', fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, df_cancel['Passengers']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 10,
            f'{int(val)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Refund Requests by Reason
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.bar(range(len(df_cancel)), df_cancel['Refund_Requests'],
               color=colors1, alpha=0.8)
ax2.set_xticks(range(len(df_cancel)))
ax2.set_xticklabels(df_cancel['Reason'], rotation=45, ha='right', fontsize=9)
ax2.set_title('Refund Requests by Reason', fontsize=13, fontweight='bold')
ax2.set_ylabel('Number of Refunds', fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, df_cancel['Refund_Requests']):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 3,
            f'{int(val)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 3: Refund Rate Percentage
ax3 = fig.add_subplot(gs[1, 0])
bars3 = ax3.barh(range(len(df_cancel)), df_cancel['Refund_Pct'],
                color=colors1, alpha=0.8)
ax3.set_yticks(range(len(df_cancel)))
ax3.set_yticklabels(df_cancel['Reason'], fontsize=10)
ax3.set_title('Refund Rate by Reason', fontsize=13, fontweight='bold')
ax3.set_xlabel('Refund Rate (%)', fontsize=11)
ax3.grid(True, alpha=0.3, axis='x')
ax3.invert_yaxis()
for bar, val in zip(bars3, df_cancel['Refund_Pct']):
    ax3.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

# Plot 4: Total Refund Amount
ax4 = fig.add_subplot(gs[1, 1])
bars4 = ax4.bar(range(len(df_cancel)), df_cancel['Total_Refunds'],
               color=colors1, alpha=0.8)
ax4.set_xticks(range(len(df_cancel)))
ax4.set_xticklabels(df_cancel['Reason'], rotation=45, ha='right', fontsize=9)
ax4.set_title('Total Refund Amount by Reason', fontsize=13, fontweight='bold')
ax4.set_ylabel('Refund Amount (£)', fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars4, df_cancel['Total_Refunds']):
    ax4.text(bar.get_x() + bar.get_width()/2, val + 60,
            f'£{int(val):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get journey status data
df_status = spark.sql("""
SELECT 
    Journey_Status,
    COUNT(*) AS Services,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS Percentage
FROM uk_train_ride.train_rise.railway
GROUP BY Journey_Status
""").toPandas()

# Create pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Journey Status Analysis', fontsize=16, fontweight='bold')

# Pie Chart
colors_pie = [COLORS['tertiary'], COLORS['secondary'], COLORS['quaternary']]
wedges, texts, autotexts = ax1.pie(df_status['Services'], 
                                    labels=df_status['Journey_Status'],
                                    autopct='%1.1f%%',
                                    colors=colors_pie,
                                    startangle=90,
                                    textprops={'fontsize': 12})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax1.set_title('Service Distribution', fontsize=14, fontweight='bold')

# Bar Chart
colors_bar = [COLORS['tertiary'], COLORS['secondary'], COLORS['quaternary']]
bars = ax2.bar(df_status['Journey_Status'], df_status['Services'],
              color=colors_bar, alpha=0.8, width=0.5)
ax2.set_title('Service Count by Status', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Services', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')
for bar, val, pct in zip(bars, df_status['Services'], df_status['Percentage']):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 300,
            f'{int(val):,}\n({pct}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get delay reasons data
df_delay = spark.sql("""
SELECT 
    Reason_for_Delay,
    COUNT(*) AS Number_of_Passengers,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS Percentage
FROM uk_train_ride.train_rise.railway
WHERE Journey_Status = 'Delayed' 
GROUP BY Reason_for_Delay
ORDER BY Number_of_Passengers DESC
""").toPandas()

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 7))
colors = [COLORS['secondary'], COLORS['quaternary'], COLORS['primary'], 
          COLORS['purple'], COLORS['brown']]
bars = ax.barh(range(len(df_delay)), df_delay['Number_of_Passengers'],
              color=colors, alpha=0.8)
ax.set_yticks(range(len(df_delay)))
ax.set_yticklabels(df_delay['Reason_for_Delay'], fontsize=11)
ax.set_title('Reasons for Service Delays', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Number of Passengers Affected', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Add value labels
for bar, val, pct in zip(bars, df_delay['Number_of_Passengers'], df_delay['Percentage']):
    ax.text(val + 20, bar.get_y() + bar.get_height()/2,
            f'{int(val):,} ({pct}%)',
            va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
%sql

SELECT 
Journey_Status
,COUNT(*) AS Number_of_Servies 
,ROUND(COUNT(*)/SUM(COUNT(*)) OVER()* 100,2) AS Percentage
FROM railway
GROUP BY Journey_Status;

In [0]:
%sql
WITH cancelled_trains AS (
SELECT 
    MONTH(Date_of_Journey) AS Month_num
    ,MONTHNAME(Date_of_Journey) AS Months
    ,SUM(
        CASE WHEN Journey_Status = 'Delayed' THEN 1 ELSE 0 END
        ) AS Delayed_Servies
    ,LAG(COUNT(Journey_Status)) OVER(ORDER BY MONTH(Date_of_Journey)) AS Previous_Month
FROM railway
WHERE Journey_Status = 'Delayed'
GROUP BY Month_num,Months
)
SELECT 
Months,
Delayed_Servies,
ROUND(
        (Delayed_Servies - previous_month)/Previous_Month  * 100 ,
         
         2) AS MoM_Change
FROM cancelled_trains; 

Delayed services remained relatively stable between January and February before increasing sharply by 14.44% in March. However, April saw a significant 17.45% reduction in delays, bringing delayed services to 530—the lowest level across the four-month period. This suggests a potential improvement in operational performance following the deterioration observed in March.

-- BEGIN TRANSCATION;

-- UPDATE railway
-- SET Reason_for_Delay = 'Signal Failure' 
--WHERE Reason_for_Delay = 'Signal failure';

-- UPDATE railway
-- SET Reason_for_Delay = 'Weather Conditions' 
--- WHERE Reason_for_Delay = 'Weather Condition';

--UPDATE railway
-- SET Reason_for_Delay = 'Staff Shortage' 
-- WHERE Reason_for_Delay = 'Staffing';

--SELECT * FROM railway;

-- COMMIT;


In [0]:
%sql
SELECT 
Reason_for_Delay
,COUNT(*) Number_of_Passengers
,ROUND(COUNT(*)/SUM(COUNT(*)) OVER() * 100,2) AS Percentage
FROM railway
WHERE Journey_Status = 'Delayed' 
GROUP BY Reason_for_Delay;



In [0]:
%sql
SELECT
    MONTH(Date_of_Journey) AS Monthnum,
    MONTHNAME(Date_of_Journey) AS Month,
    SUM(CASE 
        WHEN Reason_for_Delay LIKE '%Weather Condition%' THEN 1 
        ELSE 0 
    END) AS Weather_Delays,
    COUNT(*) AS Total_Journeys,
    ROUND(
        SUM(CASE 
            WHEN Reason_for_Delay LIKE '%Weather Condition%' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*),
        2
    ) AS Weather_Delay_Rate
FROM railway
GROUP BY
    MONTH(Date_of_Journey),
    MONTHNAME(Date_of_Journey)
ORDER BY Monthnum;

In [0]:
%sql
WITH Total_travel AS (
    SELECT 
        MONTH(Date_of_Journey) AS Month_num
        ,MONTHNAME(Date_of_Journey) AS Months
        ,COUNT(*) AS Total
    FROM railway
    GROUP BY Month_num,Months
),
Ontime(
SELECT 
    MONTH(Date_of_Journey) AS Month_num
    ,MONTHNAME(Date_of_Journey) AS Months
    ,SUM(CASE WHEN Journey_Status like '%On Time%' THEN 1 ELSE 0 END) AS On_Time
FROM railway
GROUP BY Month_num,Months
)
SELECT 
T.months
,T.Total 
,O.On_Time
,ROUND(( O.On_Time/T.Total) *100,2) AS On_Time_Percentage
FROM Total_travel T
JOIN Ontime O
ON T.Month_num = O.Month_num
ORDER BY T.month_num;
   

Key Insights:

- January had the highest consistency at 87.25% on-time performance.
- March had the lowest consistency at 85.99%, representing a drop of about 1.26 percentage points from January.
- Despite March having the highest total number of services (8,117), it had the worst on-time performance.
- April showed improvement from March, returning to approximately 87% on-time rate.